# SIH26119 — MILP iteration 04: checked candidate repair

1. Import this small notebook (under 1 MB).
2. Upload the matching **SIH26119_V2_source.zip** as a **private Kaggle dataset**
   and attach it using **Add Input**. Any dataset name works; zipped/unpacked
   layouts are detected automatically.
3. Enable **Internet**. **GPU is not needed**: this MILP implementation runs on CPU.
4. **Run All**, or **Save Version → Save & Run All**. Allow about **20-25 minutes**.
5. Download **sih26119_milp_results.zip** from Output and send it back.

Compares the recovery + neighbourhood baseline against the same solver
with an opt-in dense candidate-repair fallback
on blend2, binkar10_1, and markshare_4_0. Each gets the SAME 180-second and
250,000-node limits. All failures remain visible, and candidates are independently
checked. A shorter local smoke run passed the generated correctness gate; hard
case performance at these longer limits is not yet established.

There are only two code cells. No manual path edits or second environment.
This is separate from the previous QP/MILP notebooks and does not run the GPU
scale sweep. The companion ZIP is input data, outside the notebook size limit.

240 generated checks cover the baseline, repair mode and legacy control.
Both hard-run modes include the shared proof-safety fix: repairing a nearly
integer point does not prove that the entire node is solved. Rejected points,
fixed-integer LP results, fallback counts and checked incumbent traces are
saved in the output ZIP. Feasibility/integrality tolerances are unchanged.


In [ ]:
# 1. Load the companion source ZIP attached as a Kaggle input dataset.
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, traceback, zipfile
from IPython.display import display, FileLink

BASE = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
INPUT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else BASE/"input"
WORK = Path(tempfile.mkdtemp(prefix="sih26119_", dir=BASE))
PYTHON = sys.executable
SNAPSHOT_SHA256 = "402e0c51b9684140cc7f7d41fc2cd44aea197c2f54b4e17b9829c225765dfab2"
MANIFEST_SHA256 = "2d929d71c5683d4d4b246e78836eeca8707a1563237567b94391ec12bd01f0c1"
loaded = False

# Accept either the uploaded ZIP or files already unpacked by the dataset upload.
for source_zip in sorted(INPUT.rglob("SIH26119_V2_source.zip")):
    if hashlib.sha256(source_zip.read_bytes()).hexdigest() != SNAPSHOT_SHA256:
        continue
    with zipfile.ZipFile(source_zip) as archive:
        for entry in archive.infolist():
            if WORK.resolve() not in (WORK/entry.filename).resolve().parents:
                raise RuntimeError("Unsafe archive path")
        archive.extractall(WORK)
    loaded = True
    break
if not loaded:
    for marker in sorted(INPUT.rglob("SOURCE_MANIFEST.json")):
        if hashlib.sha256(marker.read_bytes()).hexdigest() != MANIFEST_SHA256:
            continue
        source_root = marker.parent.resolve()
        for name in json.loads(marker.read_text(encoding="utf-8")):
            source = (source_root/name).resolve()
            target = (WORK/name).resolve()
            if source_root not in source.parents or WORK.resolve() not in target.parents:
                raise RuntimeError("Unsafe dataset path")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
        shutil.copy2(marker, WORK/"SOURCE_MANIFEST.json")
        loaded = True
        break
if not loaded:
    raise FileNotFoundError(
        "Attach the NEW SIH26119_V2_source.zip as a Kaggle input dataset, then Run All. "
        "The dataset name does not matter. An older ZIP will not match this notebook."
    )

manifest_path = WORK/"SOURCE_MANIFEST.json"
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == MANIFEST_SHA256
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for name, expected in manifest.items():
    assert hashlib.sha256((WORK/name).read_bytes()).hexdigest() == expected, "Source changed: "+name
(WORK/"SNAPSHOT.json").write_text(json.dumps({"snapshot_sha256": SNAPSHOT_SHA256}, indent=2))
os.chdir(WORK)
assert Path("scripts/test_result_parser.py").is_file()
print("Working directory:", Path.cwd())
print("Python:", PYTHON)
print("Source verified. Now run the next cell.")


## Run and get the ZIP

The following cell installs comparison packages in the current Python, builds
the solver, and runs the checks. Leave it running until it prints the ZIP path.
Known hard benchmark timeouts remain in the results; they are not hidden.


In [ ]:
# 2. Focused hard-MILP comparison. CPU only; no GPU quota required.
MILP_ONLY = True
EXPERIMENT = "repair"
SECONDS_PER_CASE = 180
NODE_LIMIT = 250000
REFERENCE_SECONDS = 30

RUN = Path(tempfile.mkdtemp(prefix="milp_run_", dir=WORK))
LOGS = RUN/"run_logs"
LOGS.mkdir()
RESULTS = RUN/"reports"/"milp_focus"
stage = "setup"
status = {"completed": False, "gpu_enabled": False, "profile": "milp_focus", "experiment": EXPERIMENT, "python": PYTHON}

try:
    print("CPU-only MILP experiment: 3 models x 2 strategies x 180 seconds maximum.", flush=True)
    print("Experiment:", EXPERIMENT, flush=True)
    print("Allow about 20-25 minutes including installation, checks and reference runs.", flush=True)
    commands = [
        ("install", [PYTHON, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                     "highspy==1.12.0", "numpy", "psutil", "certifi"]),
        ("versions", [PYTHON, "-m", "pip", "freeze"]),
        ("compiler", ["g++", "--version"]),
        ("configure", ["cmake", "-S", str(WORK), "-B", str(WORK/"build-milp-focus"),
                       "-DCMAKE_BUILD_TYPE=Release", "-DSOVEREIGN_ENABLE_CUDA=OFF"]),
        ("build", ["cmake", "--build", str(WORK/"build-milp-focus"), "-j", "2"]),
        ("parser_tests", [PYTHON, str(WORK/"scripts/test_result_parser.py")]),
        ("validation", [PYTHON, str(WORK/"scripts/validate_milp_focus.py"), "--output", str(RESULTS),
                        "--limit", str(SECONDS_PER_CASE), "--nodes", str(NODE_LIMIT),
                        "--reference-limit", str(REFERENCE_SECONDS), "--experiment", EXPERIMENT]),
    ]
    for stage, command in commands:
        print("Running:", stage, flush=True)
        log_path = LOGS/(stage+".log")
        with log_path.open("w", encoding="utf-8") as log:
            result = subprocess.run(command, cwd=WORK, stdout=log, stderr=subprocess.STDOUT, timeout=3600)
        if stage != "versions":
            print(log_path.read_text(encoding="utf-8", errors="replace")[-6000:], flush=True)
        if result.returncode != 0:
            raise RuntimeError(f"{stage} exited with code {result.returncode}; see run_logs/{stage}.log")
    summary = json.loads((RESULTS/"summary.json").read_text())
    status["completed"] = True
    print(json.dumps(summary, indent=2))
    print("Comparison finished. Unsolved hard cases remain in the results.")
except Exception:
    status["failed_stage"] = stage
    status["error"] = traceback.format_exc()
    print(status["error"])
    print("Send the ZIP below even if this run failed.")
finally:
    (LOGS/"RUN_STATUS.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    destination = BASE/"sih26119_milp_results.zip"
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        for folder in (LOGS, RESULTS):
            if folder.exists():
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        archive.write(path, path.relative_to(RUN).as_posix())
        for name in ("SOURCE_MANIFEST.json", "SNAPSHOT.json"):
            if (WORK/name).is_file():
                archive.write(WORK/name, name)
    os.chdir(BASE)
    print("DOWNLOAD AND SEND BACK:", destination)
    display(FileLink(destination.name))
